# 07e · Weather: ERA5-Land from Google Earth Engine

The same `era5_land` table as notebook 07 (sensible heat flux, sunshine, ground temperature, top soil moisture, albedo, snow cover at 0.1°, about 9 km), read from **Google Earth Engine** (`ECMWF/ERA5_LAND/HOURLY`) instead of the CDS queue: hours instead of days.

One request per site and competition day (and the day before): the ERA5-Land points of the site's chips, every hour from 5 to 21 h local solar time. Each month is saved to `database/weather/gee/land/` as soon as it is read, so **if Colab disconnects, Run All again**. Whatever notebook 07 already downloaded from the CDS is merged in the last step; the same hour read twice is kept once.

Needs `GEE_PROJECT` in the settings: the Google Cloud project you use with Earth Engine.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
GEE_PROJECT = ""   # your Google Cloud project registered for Earth Engine, e.g. "ee-roberto"
THREADS = 16      # requests to Earth Engine at a time
SOLAR = "5-21"    # local solar hours kept for each site
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, get the code

In [ ]:
import pandas as pd
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "site_chips.parquet")), "Run notebook 06 first: database/site_chips.parquet is missing"
subprocess.run(["pip", "install", "-q", "earthengine-api", "netCDF4"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script, and a script
    that fails stops Run All."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise
    if code != 0:
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with an error (exit code {code}); see the lines above")
    return code

## 3. Earth Engine login

Choose the Google account that has Earth Engine. The check reads one value.

In [ ]:
import ee
assert GEE_PROJECT, "Write your Earth Engine project in GEE_PROJECT (step 1)"
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print("Earth Engine OK:", ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY").first().bandNames().getInfo())

## 4. What there is to read

In [ ]:
run(f"{REPO}/scripts/era5_land_gee.py", DB, "--project", GEE_PROJECT, "--solar", SOLAR, "--plan-only")

## 5. Read

One line per month with the time left. Requests that fail are retried on the next run.

In [ ]:
run(f"{REPO}/scripts/era5_land_gee.py", DB, "--project", GEE_PROJECT, "--threads", str(THREADS), "--solar", SOLAR)

## 6. Build the table

Merges these months with whatever notebook 07 downloaded from the CDS into `weather/era5_land.parquet`.

In [ ]:
run(f"{REPO}/scripts/era5.py", DB, "--convert-only", "--tables", "land")

## 7. A look at the table

In [ ]:
w = pd.read_parquet(os.path.join(DB, "weather", "era5_land.parquet"))
print(f"{len(w):,} rows, {w.chip_id.nunique():,} chips, {len(w[['lat', 'lon']].drop_duplicates()):,} points, {w.time.dt.normalize().nunique():,} days")
print(w.drop(columns=["chip_id", "time", "lat", "lon"]).describe().T[["mean", "min", "50%", "max"]].round(2))